In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Load Dataset

In [ ]:
import random
from torch.utils.data import Subset

data_dir = "./imagenet_validation"      # Download from https://www.kaggle.com/datasets/tusonggao/imagenet-validation-dataset

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

n_samples = 5000
indices = random.sample(range(len(dataset)), n_samples)
subset = Subset(dataset, indices)

loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

num_classes = len(dataset.classes)
print("Dataset size:", len(loader.dataset))
print("Classes:", num_classes)

Dataset size: 5000
Classes: 1000


# Evaluate Models

In [4]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoProcessor


class ModelWrapper(nn.Module):
    def __init__(self, model_name, num_classes, device):
        super().__init__()
        self.name = model_name
        self.device = device

        # ===== ViT (timm supervised) =====
        if model_name == "vit_base":
            self.model = timm.create_model(
                "vit_base_patch16_224", pretrained=True, num_classes=num_classes
            )
            self.processor = None
            self.is_feature_model = False
        elif model_name == "vit_large":
            self.model = timm.create_model(
                "vit_large_patch16_224", pretrained=True, num_classes=num_classes
            )
            self.processor = None
            self.is_feature_model = False

        # ===== CLIP =====
        # elif model_name == "clip_base":
        #     self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
        #     self.processor = CLIPProcessor.from_pretrained(
        #         "openai/clip-vit-base-patch16"
        #     )
        #     self.is_feature_model = True
        #     self.classifier = nn.Linear(512, num_classes)
        # elif model_name == "clip_large":
        #     self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
        #     self.processor = CLIPProcessor.from_pretrained(
        #         "openai/clip-vit-large-patch14"
        #     )
        #     self.is_feature_model = True
        #     self.classifier = nn.Linear(1024, num_classes)

        # ===== DINOv2 =====
        elif model_name == "dinov2_base":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_lc")
            self.processor = None
            self.is_feature_model = False
        elif model_name == "dinov2_large":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14_lc")
            self.processor = None
            self.is_feature_model = False

        # ===== SigLIP =====
        # elif model_name == "siglip":
        #     self.model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
        #     self.processor = AutoProcessor.from_pretrained(
        #         "google/siglip-base-patch16-224"
        #     )
        #     self.is_feature_model = True
        #     self.classifier = nn.Linear(768, num_classes)

        else:
            raise ValueError(f"Unknown model: {model_name}")

        self.model = self.model.to(device)
        if self.is_feature_model:
            self.classifier = self.classifier.to(device)

        self.eval()

    def forward(self, images):
        with torch.no_grad():

            # ===== ViT =====
            if not self.is_feature_model:
                return self.model(images)

            # ===== CLIP =====
            # if self.name == "clip_base" or self.name == "clip_large":
            #     inputs = self.processor(images=images, return_tensors="pt").to(
            #         self.device
            #     )
            #     features = self.model.get_image_features(**inputs)

            # ===== DINOv2 =====
            elif self.name == "dinov2_base" or self.name == "dinov2_large":
                return self.model(images)

            # ===== SigLIP =====
            # elif self.name == "siglip":
            #     inputs = self.processor(images=images, return_tensors="pt").to(
            #         self.device
            #     )
            #     outputs = self.model(**inputs)
            #     features = outputs.last_hidden_state[:, 0]

            # Ensure tensor
            # if not isinstance(features, torch.Tensor):
            #     raise TypeError(f"Expected tensor, got {type(features)}")

            # logits = self.classifier(features)
            # return logits

In [5]:
from pathlib import Path
import torchvision.transforms.functional as TF


def apply_affine_batch(images, angle, translate, scale, shear):
    return torch.stack(
        [
            TF.affine(img, angle=angle, translate=translate, scale=scale, shear=shear)
            for img in images
        ]
    )


def evaluate(
    model,
    loader,
    model_name: str = "Model",
    translation: tuple[float, float] = [0, 0],
    rotation: float = 0.0,
    scale: float = 1.0,
    shear: tuple[float, float] = [0, 0],
    save_results_path: str = "results.csv",
):
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = apply_affine_batch(
                images, angle=rotation, translate=translation, scale=scale, shear=shear
            )

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds_top1 = outputs.argmax(dim=1)
            correct_top1 += (preds_top1 == labels).sum().item()

            _, preds_top5 = outputs.topk(5, dim=1)
            correct_top5 += (preds_top5 == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    acc_top1 = correct_top1 / total
    acc_top5 = correct_top5 / total

    results_path = Path(save_results_path)
    header = (
        "Model Name,Dataset Size,Translation X, Translation Y,"
        "Rotation,Scale,Shear X, Shear Y,Top-1 Accuracy,Top-5 Accuracy"
    )

    if not results_path.exists() or results_path.stat().st_size == 0:
        # file doesn't exist or is empty -> create with header
        with results_path.open("w", encoding="utf-8") as f:
            f.write(header + "\n")
    else:
        # file exists, check header consistency
        with results_path.open("r", encoding="utf-8") as f:
            first_line = f.readline().strip()
        if first_line != header:
            with results_path.open("w", encoding="utf-8") as f:
                f.write(header + "\n")

    # append result row
    with results_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{model_name},{len(loader.dataset)},"
            f"{translation[0]},{translation[1]},{rotation},{scale},"
            f"{shear[0]},{shear[1]},{acc_top1:.4f},{acc_top5:.4f}\n"
        )

    return acc_top1, acc_top5

In [6]:
def evaluate_architecture(model_name, loader, geometric_transforms):
    model = ModelWrapper(model_name, num_classes, device)

    # Baseline evaluation without transformations
    if geometric_transforms["baseline"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name)
        print(f"Baseline (No Transformations):")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Translations
    for translation in geometric_transforms["translations"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, translation=translation)
        print(f"Translation {translation}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Rotations
    for rotation in geometric_transforms["rotations"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, rotation=rotation)
        print(f"Rotation {rotation}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Scales
    for scale in geometric_transforms["scales"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, scale=scale)
        print(f"Scale {scale}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Shears
    for shear in geometric_transforms["shears"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, shear=shear)
        print(f"Shear {shear}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

### ViT Base

In [ ]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_base", loader, geometric_transforms)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  0%|          | 0/79 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100%|██████████| 79/79 [01:03<00:00,  1.25it/s]


Baseline (No Transformations):
Top-1 Accuracy: 81.60%
Top-5 Accuracy: 95.66%


100%|██████████| 79/79 [01:08<00:00,  1.15it/s]


...

Shear (0, 10):
Top-1 Accuracy: 79.80%
Top-5 Accuracy: 95.02%


100%|██████████| 79/79 [01:11<00:00,  1.11it/s]


Shear (-10, 0):
Top-1 Accuracy: 79.24%
Top-5 Accuracy: 95.02%


100%|██████████| 79/79 [01:11<00:00,  1.10it/s]

Shear (0, -10):
Top-1 Accuracy: 79.44%
Top-5 Accuracy: 95.02%


In [ ]:
evaluate_architecture("vit_base_patch16_224", loader, geometric_transforms)

  0%|          | 0/79 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100%|██████████| 79/79 [01:11<00:00,  1.10it/s]


Translation (25, 0):
Top-1 Accuracy: 80.68%
Top-5 Accuracy: 95.64%


100%|██████████| 79/79 [01:12<00:00,  1.09it/s]


Translation (0, 25):
Top-1 Accuracy: 80.82%
Top-5 Accuracy: 95.38%


100%|██████████| 79/79 [01:12<00:00,  1.10it/s]


...

Shear (0, 50):
Top-1 Accuracy: 57.94%
Top-5 Accuracy: 79.82%


100%|██████████| 79/79 [01:11<00:00,  1.10it/s]


Shear (-50, 0):
Top-1 Accuracy: 57.16%
Top-5 Accuracy: 78.04%


100%|██████████| 79/79 [01:13<00:00,  1.08it/s]

Shear (0, -50):
Top-1 Accuracy: 58.96%
Top-5 Accuracy: 80.94%


### ViT Large

In [ ]:
geometric_transforms = {
    # "baseline": True,
    "baseline": False,
    "translations": [
        # (25, 0),  # Translate 25 pixels along X-axis
        # (0, 25),  # Translate 25 pixels along Y-axis
        # (-25, 0),  # Translate -25 pixels along X-axis
        # (0, -25),  # Translate -25 pixels along Y-axis
        # (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_large_patch16_224", loader, geometric_transforms)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
  0%|          | 0/79 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self

Translation (0, 50):
Top-1 Accuracy: 82.38%
Top-5 Accuracy: 95.92%


100%|██████████| 79/79 [03:03<00:00,  2.33s/it]


Translation (-50, 0):
Top-1 Accuracy: 82.48%
Top-5 Accuracy: 96.44%


 14%|█▍        | 11/79 [00:27<02:40,  2.35s/it]

### CLIP ViT Base (fail)

In [6]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("clip_base", loader, geometric_transforms)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
  0%|          | 0/79 [00:19<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x768 and 512x1000)

### DINOv2 Base

In [7]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("dinov2_base", loader, geometric_transforms)

Using cache found in C:\Users\ericr/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\ericr/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\ericr/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\ericr/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_linear4_head.pth" to C:\Users\ericr/.cache\torch\hub\checkpoints\dinov2_vitb14_linear4_head.pth


100%|██████████| 14.7M/14.7M [00:00<00:00, 15.9MB/s]
  3%|▎         | 2/79 [00:34<22:17, 17.37s/it]


KeyboardInterrupt: 

### DINOv2 Large

In [ ]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("dinov2_large", loader, geometric_transforms)